# イオン化ポテンシャル (IP) と ホール再配列エネルギー (λ_h) の計算
指定したSMILESから、以下の4つの状態のエネルギーを計算し、垂直IP (vIP)、断熱IP (aIP)、および再配列エネルギーを算出します。

* **E(N@N)**: 中性状態での最適化構造における中性状態のエネルギー
* **E(C@N)**: 中性状態での最適化構造におけるカチオン(ラジカル)状態のエネルギー
* **E(C@C)**: カチオン状態での最適化構造におけるカチオン(ラジカル)状態のエネルギー
* **E(N@C)**: カチオン状態での最適化構造における中性状態のエネルギー

**計算式**:
* 垂直IP (vIP) = E(C@N) - E(N@N)
* 断熱IP (aIP) = E(C@C) - E(N@N)
* 緩和エネルギー1 (λ1) = E(C@N) - E(C@C)
* 緩和エネルギー2 (λ2) = E(N@C) - E(N@N)
* ホール再配列エネルギー (λ_h) = λ1 + λ2


In [1]:
import os
import cupy
import gc
import time
from rdkit import Chem
from rdkit.Chem import AllChem
from pyscf import gto
from gpu4pyscf.dft import rks, uks
from pyscf.geomopt.geometric_solver import kernel as geom_kernel

# --- 設定 ---
SMILES = 'C=1(C=2C(C=CC1)=CC=CC2)N(C=3C=CC=CC3)C=4C=CC(C=5C=CC(N(C=6C=7C(C=CC6)=CC=CC7)C=8C=CC=CC8)=CC5)=CC4' # NPD
MOL_NAME = 'NPD_IP'
XC_FUNCTIONAL = 'b3lyp'
BASIS_SET = '6-31+g*' # 6-31+G(d)

out_dir = f"output/{MOL_NAME}"
os.makedirs(out_dir, exist_ok=True)

def release_gpu():
    """計算間にGPUメモリプールを返す。"""
    gc.collect()
    cupy.get_default_memory_pool().free_all_blocks()
    cupy.get_default_pinned_memory_pool().free_all_blocks()


In [2]:
# 1. RDKitによる初期構造生成
print(f"[{MOL_NAME}] RDKitで3D構造を生成中...")
mol_rdkit = Chem.MolFromSmiles(SMILES)
if mol_rdkit is None:
    raise ValueError("SMILESを解釈できませんでした。")
mol_rdkit = Chem.AddHs(mol_rdkit)
if AllChem.EmbedMolecule(mol_rdkit, randomSeed=42) == -1:
    raise ValueError("RDKitによる3D構造生成に失敗しました。")
# MMFFが組めないときは例外ではなく -1。そのときだけUFFを使う
try:
    mmff_failed = AllChem.MMFFOptimizeMolecule(mol_rdkit) == -1
except Exception:
    mmff_failed = True
if mmff_failed:
    AllChem.UFFOptimizeMolecule(mol_rdkit)

rdkit_xyz = os.path.join(out_dir, f"{MOL_NAME}_rdkit.xyz")
Chem.MolToXYZFile(mol_rdkit, rdkit_xyz)
print("RDKit構造を保存しました。")


[NPD_IP] RDKitで3D構造を生成中...
RDKit構造を保存しました。


In [3]:
# 2. 中性状態の構造最適化 (N@N)
print("\n--- [1/4] 中性状態の構造最適化 (N@N) ---")
mol_N = gto.M(atom=rdkit_xyz, basis=BASIS_SET, charge=0, spin=0)
mf_N = rks.RKS(mol_N, xc=XC_FUNCTIONAL)

conv, mol_N_opt = geom_kernel(mf_N, maxsteps=100)
if not conv:
    raise RuntimeError("中性状態の構造最適化が100ステップ以内に収束しませんでした。")
del mf_N
release_gpu()

# 最適化構造でのエネルギーを取得するため、再度SP計算を実行。
# density_fit の3中心積分は分子が大きくなると GPU (8GB) に載らない。
# 構造最適化と同じ直接SCFを使う。
mf_N_opt = rks.RKS(mol_N_opt, xc=XC_FUNCTIONAL)
E_N_N = mf_N_opt.kernel()
print(f"E(N@N) = {E_N_N:.6f} Hartree")

del mf_N_opt
release_gpu()

# 中性最適化構造の保存
xyz_N_opt = os.path.join(out_dir, f"{MOL_NAME}_Neutral_opt.xyz")
with open(xyz_N_opt, "w") as f:
    f.write(mol_N_opt.tostring(format='xyz'))


geometric-optimize called with the following command line:
/home/kawano/miniforge3/envs/gpu4pyscf/lib/python3.10/site-packages/ipykernel_launcher.py --f=/mnt/wslg/runtime-dir/jupyter/runtime/kernel-v3517726147ee69aa597a9f0d4e6d5aff313e2cecd.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%

              *%%%%%%/      (%%%%%%   ********/*..,*/*********       *))))))   
                #%%/      (%%%%%%,    *********************/        )))))))    
                        *%%%%%%,         ,**************/         ,))))))/     
                      (%%%%%%   ()                              ))))))))       
                      #%%%%,  ())))))                        ,)))))))),        
                        #,    ())))))))))                ,)))))))))).          
                                 ()))))))))))))))))))))))))))))))/             
                                    ())))))))))))))))))))))))).                
                                         ())))))))))))))),                     

-=#  geomeTRIC started. Version: 1.1.1  #=-
Current date and time: 2026-10-02 09:12:49
#========================================================#
#|     Arguments passed to driver run_optimizer():      |#
#========================================================#
customengine   


--- [1/4] 中性状態の構造最適化 (N@N) ---


234 internal coordinates being used (instead of 234 Cartesians)
Internal coordinate system (atoms numbered from 1):
Distance 1-2
Distance 1-6
Distance 1-11
Distance 2-3
Distance 2-10
Distance 3-4
Distance 3-7
Distance 4-5
Distance 4-47
Distance 5-6
Distance 5-48
Distance 6-49
Distance 7-8
Distance 7-50
Distance 8-9
Distance 8-51
Distance 9-10
Distance 9-52
Distance 10-53
Distance 11-12
Distance 11-18
Distance 12-13
Distance 12-17
Distance 13-14
Distance 13-54
Distance 14-15
Distance 14-55
Distance 15-16
Distance 15-56
Distance 16-17
Distance 16-57
Distance 17-58
Distance 18-19
Distance 18-46
Distance 19-20
Distance 19-59
Distance 20-21
Distance 20-60
Distance 21-22
Distance 21-45
Distance 22-23
Distance 22-44
Distance 23-24
Distance 23-61
Distance 24-25
Distance 24-62
Distance 25-26
Distance 25-43
Distance 26-27
Distance 26-37
Distance 27-28
Distance 27-32
Distance 28-29
Distance 28-36
Distance 29-30
Distance 29-33
Distance 30-31
Distance 30-63
Distance 31-32
Distance 31-64
Distance 32


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C  -5.433480   1.190742  -0.200437    0.000000  0.000000  0.000000
   C  -6.253724   1.324531  -1.361917    0.000000  0.000000  0.000000
   C  -6.840053   2.587142  -1.634775    0.000000 -0.000000  0.000000
   C  -6.597849   3.673643  -0.779648    0.000000  0.000000  0.000000
   C  -5.779366   3.540219   0.338404    0.000000  0.000000  0.000000
   C  -5.199944   2.309755   0.624773    0.000000  0.000000  0.000000
   C  -7.654987   2.754616  -2.766844    0.000000  0.000000  0.000000
   C  -7.891343   1.698621  -3.644929    0.000000  0.000000  0.000000
   C  -7.309408   0.461561  -3.406424    0.000000  0.000000  0.000000
   C  -6.499031   0.279566  -2.283895    0.000000  0.000000  0.000000
   N  -4.823365  -0.071356   0.172430    0.000000  0.000000  0.000000
   C  -5.663747  -1.191871   0.527488    0.000000  0.000000  0.000000
   C  -6.932564  -0.999587   1.1

...


Geometry optimization cycle 27
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C  -5.519319   1.056133   0.206394    0.000439  0.000545  0.000221
   C  -6.403490   1.606644  -0.781012    0.000826  0.000378 -0.000215
   C  -7.114033   2.814579  -0.466173    0.000937  0.000623 -0.000889
   C  -6.904373   3.445791   0.790688    0.000808  0.000900 -0.001010
   C  -6.024848   2.908530   1.705417    0.000451  0.001029 -0.000585
   C  -5.338262   1.707131   1.414360    0.000337  0.000803  0.000099
   C  -7.998394   3.365865  -1.435392    0.001310  0.000528 -0.001290
   C  -8.166911   2.768063  -2.665390    0.001611  0.000101 -0.001117
   C  -7.448060   1.589013  -2.983406    0.001488 -0.000152 -0.000479
   C  -6.589914   1.022016  -2.064358    0.001105 -0.000045 -0.000062
   N  -4.806016  -0.155956  -0.061241    0.000267  0.000303  0.000849
   C  -5.520936  -1.375545  -0.189765    0.000157  0.000365  0.000844
   C  -6.761959  -1.554012   0.

Step   26 : Displace = 9.901e-04/2.291e-03 (rms/max) Trust = 9.770e-04 (-) Grad = 1.044e-05/4.122e-05 (rms/max) E (change) = -1805.5094360417 (-3.353e-08) Quality = 0.203
Hessian Eigenvalues: 4.59408e-03 5.18634e-03 1.89842e-02 ... 7.63405e-01 8.76256e-01 9.50223e-01



Geometry optimization cycle 28
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C  -5.519202   1.056408   0.206455    0.000116  0.000274  0.000062
   C  -6.403120   1.606903  -0.781182    0.000370  0.000259 -0.000170
   C  -7.113665   2.814916  -0.466623    0.000368  0.000337 -0.000450
   C  -6.904245   3.446214   0.790232    0.000129  0.000424 -0.000456
   C  -6.024969   2.908990   1.705220   -0.000120  0.000460 -0.000197
   C  -5.338406   1.707523   1.414394   -0.000145  0.000392  0.000034
   C  -7.997774   3.366196  -1.436078    0.000619  0.000331 -0.000687
   C  -8.166022   2.768271  -2.666052    0.000889  0.000208 -0.000662
   C  -7.447183   1.589142  -2.983808    0.000877  0.000130 -0.000402
   C  -6.589288   1.022166  -2.064517    0.000626  0.000151 -0.000159
   N  -4.805906  -0.155761  -0.060865    0.000110  0.000196  0.000376
   C  -5.520895  -1.375297  -0.189448    0.000041  0.000247  0.000317
   C  -6.761941  -1.553722   0.

Step   27 : Displace = 5.139e-04/1.231e-03 (rms/max) Trust = 4.885e-04 (-) Grad = 7.995e-06/2.633e-05 (rms/max) E (change) = -1805.5094360003 (+4.143e-08) Quality = -0.713
Hessian Eigenvalues: 4.59408e-03 5.18634e-03 1.89842e-02 ... 7.63405e-01 8.76256e-01 9.50223e-01
Converged! =D

    #==========================================================================#
    #| If this code has benefited your research, please support us by citing: |#
    #|                                                                        |#
    #| Wang, L.-P.; Song, C.C. (2016) "Geometry optimization made simple with |#
    #| translation and rotation coordinates", J. Chem, Phys. 144, 214108.     |#
    #| http://dx.doi.org/10.1063/1.4952956                                    |#
    #==========================================================================#
    Time elapsed since start of run_optimizer: 17324.669 seconds



WARN: 13 small eigenvectors of overlap matrix removed because of linear dependency between AOs.
The support for low-rank overlap matrix is not fully tested. Please report any bug you encountered to the developers.

converged SCF energy = -1805.50933421925
E(N@N) = -1805.509334 Hartree


In [4]:
# 3. 中性構造でのカチオン状態の単点計算 (C@N)
# 中性の構造 (mol_N_opt) を使用し、電荷+1, スピン1(Doublet) に設定
print("\n--- [2/4] 中性構造でのカチオンの単点計算 (C@N) ---")
mol_C_at_N = mol_N_opt.copy()
mol_C_at_N.charge = 1
mol_C_at_N.spin = 1
mol_C_at_N.build()

# 開殻系なので UKS を使用
mf_C_at_N = uks.UKS(mol_C_at_N, xc=XC_FUNCTIONAL)

E_C_N = mf_C_at_N.kernel()
del mf_C_at_N
release_gpu()
print(f"E(C@N) = {E_C_N:.6f} Hartree")


--- [2/4] 中性構造でのカチオンの単点計算 (C@N) ---



WARN: 13 small eigenvectors of overlap matrix removed because of linear dependency between AOs.
The support for low-rank overlap matrix is not fully tested. Please report any bug you encountered to the developers.

converged SCF energy = -1805.28884128566  <S^2> = 0.76010557  2S+1 = 2.0100802
E(C@N) = -1805.288841 Hartree


In [5]:
# 4. カチオン状態の構造最適化 (C@C)
print("\n--- [3/4] カチオン状態の構造最適化 (C@C) ---")
# mf_C_at_N を初期状態として構造最適化を開始
mf_C = uks.UKS(mol_C_at_N, xc=XC_FUNCTIONAL)
conv, mol_C_opt = geom_kernel(mf_C, maxsteps=100)
if not conv:
    raise RuntimeError("カチオン状態の構造最適化が100ステップ以内に収束しませんでした。")
del mf_C
release_gpu()

# 最適化構造でのエネルギーを取得するため、再度SP計算を実行
mf_C_opt = uks.UKS(mol_C_opt, xc=XC_FUNCTIONAL)
E_C_C = mf_C_opt.kernel()

del mf_C_opt
release_gpu()
print(f"E(C@C) = {E_C_C:.6f} Hartree")

# カチオン最適化構造の保存
xyz_C_opt = os.path.join(out_dir, f"{MOL_NAME}_Cation_opt.xyz")
with open(xyz_C_opt, "w") as f:
    f.write(mol_C_opt.tostring(format='xyz'))


geometric-optimize called with the following command line:
/home/kawano/miniforge3/envs/gpu4pyscf/lib/python3.10/site-packages/ipykernel_launcher.py --f=/mnt/wslg/runtime-dir/jupyter/runtime/kernel-v3517726147ee69aa597a9f0d4e6d5aff313e2cecd.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%


--- [3/4] カチオン状態の構造最適化 (C@C) ---


234 internal coordinates being used (instead of 234 Cartesians)
Internal coordinate system (atoms numbered from 1):
Distance 1-2
Distance 1-6
Distance 1-11
Distance 2-3
Distance 2-10
Distance 3-4
Distance 3-7
Distance 4-5
Distance 4-47
Distance 5-6
Distance 5-48
Distance 6-49
Distance 7-8
Distance 7-50
Distance 8-9
Distance 8-51
Distance 9-10
Distance 9-52
Distance 10-53
Distance 11-12
Distance 11-18
Distance 12-13
Distance 12-17
Distance 13-14
Distance 13-54
Distance 14-15
Distance 14-55
Distance 15-16
Distance 15-56
Distance 16-17
Distance 16-57
Distance 17-58
Distance 18-19
Distance 18-46
Distance 19-20
Distance 19-59
Distance 20-21
Distance 20-60
Distance 21-22
Distance 21-45
Distance 22-23
Distance 22-44
Distance 23-24
Distance 23-61
Distance 24-25
Distance 24-62
Distance 25-26
Distance 25-43
Distance 26-27
Distance 26-37
Distance 27-28
Distance 27-32
Distance 28-29
Distance 28-36
Distance 29-30
Distance 29-33
Distance 30-31
Distance 30-63
Distance 31-32
Distance 31-64
Distance 32


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C  -5.519202   1.056408   0.206455    0.000000  0.000000 -0.000000
   C  -6.403120   1.606903  -0.781182    0.000000  0.000000  0.000000
   C  -7.113665   2.814916  -0.466623    0.000000 -0.000000  0.000000
   C  -6.904245   3.446214   0.790232    0.000000  0.000000  0.000000
   C  -6.024969   2.908990   1.705220    0.000000  0.000000  0.000000
   C  -5.338406   1.707523   1.414394    0.000000  0.000000  0.000000
   C  -7.997774   3.366196  -1.436078    0.000000  0.000000  0.000000
   C  -8.166022   2.768271  -2.666052    0.000000  0.000000  0.000000
   C  -7.447183   1.589142  -2.983808    0.000000  0.000000  0.000000
   C  -6.589288   1.022166  -2.064517    0.000000  0.000000  0.000000
   N  -4.805906  -0.155761  -0.060865    0.000000  0.000000  0.000000
   C  -5.520895  -1.375297  -0.189448    0.000000  0.000000  0.000000
   C  -6.761941  -1.553722   0.4

converged SCF energy = -1805.28893998018  <S^2> = 0.76010424  2S+1 = 2.0100788
--------------- UKS_Scanner gradients ---------------
         x                y                z
0 C     0.0013322955    -0.0038311767     0.0054243530
1 C     0.0020030214     0.0021501708     0.0031207364
2 C     0.0001927090     0.0004631233     0.0006006713
3 C     0.0004229572     0.0005443063     0.0019563965
4 C    -0.0015489088     0.0018817110    -0.0021971189
5 C    -0.0034016888    -0.0021947957    -0.0024634628
6 C    -0.0001428373    -0.0006264570    -0.0010424656
7 C    -0.0001846050     0.0007023533     0.0008508505
8 C     0.0008800886    -0.0011119621     0.0004235104
9 C    -0.0010809063    -0.0003962836    -0.0017247907
10 N    -0.0091167832     0.0017025193    -0.0061249664
11 C     0.0008561744     0.0029747063     0.0004257060
12 C     0.0028117168    -0.0008469791    -0.0007840691
13 C     0.0006204552    -0.0026699136    -0.0018712541
14 C    -0.0013111468    -0.0006315947     0.000

...


Geometry optimization cycle 20
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C  -5.530108   1.004645   0.316474    0.001413  0.000149  0.000382
   C  -6.355650   1.617320  -0.681236    0.000856 -0.000702  0.000280
   C  -7.114533   2.776221  -0.299252    0.000353 -0.001045  0.000180
   C  -7.012557   3.279392   1.026445    0.000539 -0.000633  0.000004
   C  -6.188056   2.676415   1.953221    0.001121  0.000205  0.000035
   C  -5.443440   1.531888   1.594821    0.001617  0.000476  0.000266
   C  -7.940755   3.410165  -1.267881   -0.000235 -0.001806  0.000186
   C  -8.008637   2.936506  -2.560172   -0.000412 -0.002031  0.000265
   C  -7.243703   1.807239  -2.940456    0.000161 -0.001680  0.000378
   C  -6.436987   1.162150  -2.025249    0.000733 -0.000992  0.000378
   N  -4.778685  -0.184596   0.015117    0.001955  0.000491  0.000670
   C  -5.494706  -1.387268  -0.279046    0.002129  0.000371  0.000755
   C  -6.681593  -1.680195   0.

Step   19 : Displace = 1.178e-03/3.122e-03 (rms/max) Trust = 1.157e-03 (=) Grad = 1.459e-05/5.776e-05 (rms/max) E (change) = -1805.2941222314 (-1.779e-08) Quality = 0.045
Hessian Eigenvalues: 3.16176e-03 5.59410e-03 9.44424e-03 ... 8.96397e-01 9.73780e-01 1.07509e+00



Geometry optimization cycle 21
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C  -5.526900   1.004306   0.317125    0.003209 -0.000339  0.000651
   C  -6.352960   1.616506  -0.680454    0.002690 -0.000814  0.000782
   C  -7.111880   2.775428  -0.298699    0.002652 -0.000793  0.000553
   C  -7.009428   3.279222   1.026722    0.003129 -0.000170  0.000277
   C  -6.184428   2.676804   1.953424    0.003628  0.000389  0.000203
   C  -5.439824   1.532226   1.595193    0.003617  0.000339  0.000372
   C  -7.938540   3.408776  -1.267346    0.002215 -0.001389  0.000535
   C  -8.006874   2.934683  -2.559464    0.001763 -0.001823  0.000708
   C  -7.241868   1.805418  -2.939598    0.001834 -0.001822  0.000858
   C  -6.434771   1.160920  -2.024292    0.002216 -0.001231  0.000956
   N  -4.775433  -0.184931   0.016048    0.003252 -0.000335  0.000930
   C  -5.491218  -1.387709  -0.278167    0.003488 -0.000441  0.000879
   C  -6.678166  -1.680469   0.

Step   20 : Displace = 6.103e-04/1.780e-03 (rms/max) Trust = 5.787e-04 (-) Grad = 1.913e-05/5.700e-05 (rms/max) E (change) = -1805.2941221630 (+6.842e-08) Quality = -0.159
Hessian Eigenvalues: 3.16176e-03 5.59410e-03 9.44424e-03 ... 8.96397e-01 9.73780e-01 1.07509e+00
Converged! =D

    #==========================================================================#
    #| If this code has benefited your research, please support us by citing: |#
    #|                                                                        |#
    #| Wang, L.-P.; Song, C.C. (2016) "Geometry optimization made simple with |#
    #| translation and rotation coordinates", J. Chem, Phys. 144, 214108.     |#
    #| http://dx.doi.org/10.1063/1.4952956                                    |#
    #==========================================================================#
    Time elapsed since start of run_optimizer: 14181.777 seconds



WARN: 15 small eigenvectors of overlap matrix removed because of linear dependency between AOs.
The support for low-rank overlap matrix is not fully tested. Please report any bug you encountered to the developers.

converged SCF energy = -1805.29401699475  <S^2> = 0.76054668  2S+1 = 2.010519
E(C@C) = -1805.294017 Hartree


In [6]:
# 5. カチオン構造での中性状態の単点計算 (N@C)
print("\n--- [4/4] カチオン構造での中性状態の単点計算 (N@C) ---")
mol_N_at_C = mol_C_opt.copy()
mol_N_at_C.charge = 0
mol_N_at_C.spin = 0
mol_N_at_C.build()

mf_N_at_C = rks.RKS(mol_N_at_C, xc=XC_FUNCTIONAL)

E_N_C = mf_N_at_C.kernel()
del mf_N_at_C
release_gpu()
print(f"E(N@C) = {E_N_C:.6f} Hartree")



--- [4/4] カチオン構造での中性状態の単点計算 (N@C) ---



WARN: 15 small eigenvectors of overlap matrix removed because of linear dependency between AOs.
The support for low-rank overlap matrix is not fully tested. Please report any bug you encountered to the developers.

converged SCF energy = -1805.50337879497
E(N@C) = -1805.503379 Hartree


In [9]:
# 6. 結果の計算と出力
# Hartree to eV conversion factor
H2EV = 27.211386245988

vIP = (E_C_N - E_N_N) * H2EV
aIP = (E_C_C - E_N_N) * H2EV
lambda_1 = (E_C_N - E_C_C) * H2EV
lambda_2 = (E_N_C - E_N_N) * H2EV
lambda_h = lambda_1 + lambda_2

print("\n" + "="*40)
print(f"【{MOL_NAME} の 計算結果】")
print(f"Level of theory: {XC_FUNCTIONAL} / {BASIS_SET}")
print(f"E(N@N) : {E_N_N:12.6f} Hartree")
print(f"E(C@N) : {E_C_N:12.6f} Hartree")
print(f"E(C@C) : {E_C_C:12.6f} Hartree")
print(f"E(N@C) : {E_N_C:12.6f} Hartree")
print("-" * 40)
print(f"垂直IP (vIP)          : {vIP:.4f} eV")
print(f"断熱IP (aIP)          : {aIP:.4f} eV")
print(f"緩和エネルギー1 (λ1) : {lambda_1:.4f} eV")
print(f"緩和エネルギー2 (λ2) : {lambda_2:.4f} eV")
print(f"ホール再配列エネルギー: {lambda_h:.4f} eV")
print("="*40)
# 結果を result.txt に出力
result_path = os.path.join(out_dir, "result.txt")
with open(result_path, "w", encoding="utf-8") as f_res:
    f_res.write("="*40 + "\n")
    f_res.write(f"【{MOL_NAME} の IP/Reorg 計算結果】\n")
    f_res.write(f"Level of theory: {XC_FUNCTIONAL} / {BASIS_SET}\n")
    f_res.write(f"E(N@N) : {E_N_N:12.6f} Hartree\n")
    f_res.write(f"E(C@N) : {E_C_N:12.6f} Hartree\n")
    f_res.write(f"E(C@C) : {E_C_C:12.6f} Hartree\n")
    f_res.write(f"E(N@C) : {E_N_C:12.6f} Hartree\n")
    f_res.write("-" * 40 + "\n")
    f_res.write(f"垂直IP (vIP)          : {vIP:.4f} eV\n")
    f_res.write(f"断熱IP (aIP)          : {aIP:.4f} eV\n")
    f_res.write(f"緩和エネルギー1 (λ1) : {lambda_1:.4f} eV\n")
    f_res.write(f"緩和エネルギー2 (λ2) : {lambda_2:.4f} eV\n")
    f_res.write(f"ホール再配列エネルギー: {lambda_h:.4f} eV\n")
    f_res.write("="*40 + "\n")
print(f"\n結果を {result_path} に保存しました。")



【NPD_IP の 計算結果】
Level of theory: b3lyp / 6-31+g*
E(N@N) : -1805.509334 Hartree
E(C@N) : -1805.288841 Hartree
E(C@C) : -1805.294017 Hartree
E(N@C) : -1805.503379 Hartree
----------------------------------------
垂直IP (vIP)          : 5.9999 eV
断熱IP (aIP)          : 5.8591 eV
緩和エネルギー1 (λ1) : 0.1408 eV
緩和エネルギー2 (λ2) : 0.1621 eV
ホール再配列エネルギー: 0.3029 eV

結果を output/NPD_IP/result.txt に保存しました。
